# Spike Data Loading

Load functional score DMS data for three spike homologs (Delta, BA.1, BA.2),
filter and clip scores, select representative replicates, and save training data.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]

output_dir = spike["output_dir"]
experiment_conditions = spike["experiment_conditions"]
replicate_1_experiments = spike["replicate_1_experiments"]
replicate_2_experiments = spike["replicate_2_experiments"]
func_score_clip = spike["func_score_clip"]
pre_count_threshold = spike["pre_count_threshold"]
subsample_frac = spike.get("subsample_frac", None)

os.makedirs(output_dir, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Load functional score data

Read `functional_selections.csv` for each homolog and construct functional score DataFrames.

In [ ]:
func_score_data = pd.DataFrame()

for homolog in experiment_conditions:
    func_sel = (
        pd.read_csv(f"data/{homolog}/functional_selections.csv")
        .assign(
            filename=lambda x: f"data/{homolog}/"
            + x.library + "_"
            + x.preselection_sample
            + "_vs_" + x.postselection_sample
            + "_func_scores.csv"
        )
        .assign(
            func_sel_scores_df=lambda x: x.filename.apply(lambda f: pd.read_csv(f))
        )
        .assign(
            len_func_sel_scores_df=lambda x: x.func_sel_scores_df.apply(len)
        )
        .assign(homolog=homolog)
    )
    func_score_data = pd.concat([func_score_data, func_sel]).reset_index(drop=True)

func_score_data["condition"] = func_score_data.apply(
    lambda row: f"{row['homolog']}-{row['library']}".replace("-Lib", ""), axis=1
)
print(f"Loaded {len(func_score_data)} experiments")
func_score_data[["library", "replicate", "filename", "condition"]]

## Concatenate and filter

In [ ]:
func_score_df = pd.DataFrame()
for idx, row in tqdm(func_score_data.iterrows(), total=len(func_score_data)):
    mut_df_replicates = row.func_sel_scores_df.assign(
        homolog=row.homolog,
        library=row.library,
        replicate=row.replicate,
        condition=row.condition,
    )
    func_score_df = pd.concat([func_score_df, mut_df_replicates])

func_score_df = (
    func_score_df.rename({"aa_substitutions_reference": "aa_substitutions"}, axis=1)
    .reset_index(drop=True)
    .fillna("")
    .sort_values(by="condition")
)
print(f"Total variants: {len(func_score_df)}")

In [ ]:
# Filter by pre-selection count threshold
n_pre_threshold = len(func_score_df)
func_score_df.query(f"pre_count >= {pre_count_threshold}", inplace=True)
print(f"Filtered {n_pre_threshold - len(func_score_df)} variants below pre_count threshold of {pre_count_threshold}")

In [ ]:
# Keep only required columns
required_cols = ["func_score", "aa_substitutions", "condition"]
func_score_df.drop([c for c in func_score_df if c not in required_cols], axis=1, inplace=True)

In [ ]:
# Remove stop-codon wildtypes and non-numeric sites (indels)
stop_wt_vars = []
non_numeric_sites = []
for idx, row in tqdm(func_score_df.iterrows(), total=len(func_score_df)):
    for sub in row["aa_substitutions"].split():
        if sub[0] == "*":
            stop_wt_vars.append(idx)
        if not sub[-2].isnumeric():
            non_numeric_sites.append(idx)

to_drop = set.union(set(stop_wt_vars), set(non_numeric_sites))
func_score_df.drop(to_drop, inplace=True)
print(f"Dropped {len(to_drop)} variants with stop-wt or non-numeric sites")

In [ ]:
# Clip functional scores
clip_lo, clip_hi = func_score_clip
n_below = len(func_score_df.query(f"func_score < {clip_lo}"))
n_above = len(func_score_df.query(f"func_score > {clip_hi}"))
print(f"Clipping: {n_below} below {clip_lo}, {n_above} above {clip_hi}")
func_score_df = func_score_df.assign(
    func_score=func_score_df.func_score.clip(clip_lo, clip_hi)
)

## Select replicates

In [ ]:
func_score_df = pd.concat([
    (
        func_score_df.query("condition in @replicate_1_experiments")
        .replace(dict(zip(replicate_1_experiments, experiment_conditions)))
        .assign(replicate=1)
    ),
    (
        func_score_df.query("condition in @replicate_2_experiments")
        .replace(dict(zip(replicate_2_experiments, experiment_conditions)))
        .assign(replicate=2)
    ),
])
func_score_df = func_score_df.assign(
    n_subs=[len(aa_subs.split()) for aa_subs in func_score_df.aa_substitutions]
)
print(f"Training data: {len(func_score_df)} variants across {func_score_df.condition.nunique()} conditions x {func_score_df.replicate.nunique()} replicates")

## Save and optionally subsample

In [ ]:
func_score_df.to_csv(f"{output_dir}/training_functional_scores.csv", index=False)
print(f"Saved {output_dir}/training_functional_scores.csv")

In [ ]:
# Optionally subsample for fast testing
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=0)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
    print(f"Subsampled to {len(func_score_df)} rows (frac={subsample_frac})")